# Solutions

:::{admonition} Reference solutions
:class: note
Worked solutions for every exercise in [1.6-geospatial-vector-data-exercises.ipynb](1.6-geospatial-vector-data-exercises.ipynb), including the hurricane-track walkthrough.
:::

## Exercise 1: Build a GeoDataFrame

Create a GeoDataFrame of two points, `A` at (7.4, 46.9) and `B` at (8.5, 47.4), in EPSG:4326. Print the epsg code and the geometry column.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
gdf = gpd.GeoDataFrame({"name": ["A", "B"]},
                       geometry=[Point(7.4, 46.9), Point(8.5, 47.4)],
                       crs="EPSG:4326")
print(gdf.crs.to_epsg())
print(gdf.geometry.tolist())

## Exercise 2: Write and read GeoJSON

Write the GeoDataFrame from exercise 1 to `_files/points.geojson`, read it back, and confirm the shape and that the crs is preserved.

In [ ]:
from pathlib import Path

Path("_files").mkdir(exist_ok=True)
gdf.to_file("_files/points.geojson", driver="GeoJSON")
back = gpd.read_file("_files/points.geojson")
print(back.shape, back.crs.to_epsg())

## Exercise 3: Reproject

Reproject the points to EPSG:2056 and print the new epsg code and the projected coordinates of point `A` (in metres, rounded to the nearest metre).

In [ ]:
proj = gdf.to_crs(2056)
print(proj.crs.to_epsg())
print(round(proj.geometry.x.iloc[0]), round(proj.geometry.y.iloc[0]))

## Exercise 4: Measure correctly

Compute the distance between `A` and `B` in kilometres. Reproject to EPSG:2056 first, and state in a comment why measuring in EPSG:4326 would be wrong.

In [ ]:
proj = gdf.to_crs(2056)                                  # metres, not degrees
d_m = proj.geometry.iloc[0].distance(proj.geometry.iloc[1])
print(round(d_m / 1000, 2), "km")
# in EPSG:4326 .distance() subtracts angles, giving degrees, which are not a length

## Exercise 5: Spatial join

Given the polygon below (EPSG:4326), use `sjoin` with the `within` predicate to find which of the two points lie inside it.

```python
from shapely.geometry import Polygon
poly = gpd.GeoDataFrame({"zone": ["z"]},
    geometry=[Polygon([(7, 46.5), (8, 46.5), (8, 47.5), (7, 47.5)])], crs="EPSG:4326")
```

In [ ]:
from shapely.geometry import Polygon
poly = gpd.GeoDataFrame({"zone": ["z"]},
    geometry=[Polygon([(7, 46.5), (8, 46.5), (8, 47.5), (7, 47.5)])], crs="EPSG:4326")
hit = gpd.sjoin(gdf, poly, predicate="within", how="inner")
print(hit["name"].tolist())     # only A lies inside

## Exercise 6: Buffer and area

Reproject the points to EPSG:2056, buffer each by 10 km, and print the area of one buffer in km² (it should be close to the analytical value pi times 10² = 314 km²).

In [ ]:
proj = gdf.to_crs(2056)
buf = proj.buffer(10_000)
print(round(buf.area.iloc[0] / 1e6, 1), "km^2")   # ~314

## Exercise 7: Dissolve

Create a GeoDataFrame of two adjacent polygons that share the attribute `type = "flood"`, then `dissolve` by that attribute and confirm the result is a single merged polygon.

In [ ]:
from shapely.geometry import Polygon
zones = gpd.GeoDataFrame({"type": ["flood", "flood"]},
    geometry=[Polygon([(7, 47), (8, 47), (8, 48), (7, 48)]),
              Polygon([(8, 47), (9, 47), (9, 48), (8, 48)])], crs="EPSG:4326")
merged = zones.dissolve(by="type")
print(len(merged))     # 1

## Exercise 8: Hurricane track analysis


*Can you quickly find out which US states Hurricane Florence passed through using geopandas?*

Apply geopandas to read in the geospatial data, plot, and analyse the track of Hurricane Florence
from 30 August to 18 September 2018. The track is the archived advisory record from the
[National Hurricane Center](https://www.nhc.noaa.gov/); the state boundaries are a
[US Census Bureau](https://www.census.gov/geographies/mapping-files/time-series/geo/carto-boundary-file.html)
cartographic boundary file.
References:

1. [Introduction to GeoPandas](https://geopandas.org/en/stable/getting_started/introduction.html) — geopandas' official website
2. [Geopandas: an introduction](https://autogis-site.readthedocs.io/en/latest/lessons/lesson-2/geopandas-an-introduction.html) — Automating GIS Processes
3. [Use Data for Earth and Environmental Science in Open Source Python](https://www.earthdatascience.org/courses/use-data-open-source-python/)
4. [The Shapely User Manual](https://shapely.readthedocs.io/en/stable/manual.html)
5. [Geospatial Analysis with Python and R](https://kodu.ut.ee/~kmoch/geopython2020/index.html)

In [ ]:
# Pre-supplied: download and cache the two real data files.
import pooch

states_path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/gz_2010_us_040_00_5m.json",
    known_hash="sha256:7a8c022e063a34a83f35984cde6c81992ece5983f8cc4459ed02e40687739573",
    fname="us_states.geojson",
    path=pooch.os_cache("mlees"),
)
track_path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/florence.csv",
    known_hash="sha256:385691583ee41682a1c905e042c56ea362609fb04645934bdda7911c55c8b63f",
    fname="florence.csv",
    path=pooch.os_cache("mlees"),
)

**Q1)** Imports.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

**Q2)** Read the boundary file.

In [ ]:
country = gpd.read_file(states_path)
print(country.shape)

**Q3)** What is in it, and what geometries does it hold?

In [ ]:
print(country.head())
print(country.geom_type.unique())

# Both Polygon and MultiPolygon: a state with offshore islands cannot be one ring, so the
# file mixes the two. Anything that assumes pure Polygon will break on this data.

**Q4)** The lower 48 on a map.

In [ ]:
lower48 = country[country["NAME"].isin(["Alaska", "Hawaii"]) == False]
lower48.plot(figsize=(30, 20))
plt.show()

In [ ]:
# Pre-supplied: read in the hurricane Florence data, drop the advisory bookkeeping columns,
# fix the longitude sign, and have a look at the dataframe.
florence = pd.read_csv(track_path)
florence = florence.drop(["AdvisoryNumber", "Forecaster", "Received"], axis=1)
florence["Long"] = 0 - florence["Long"]
florence.head(3)

**Q5)** The track as a GeoDataFrame.

In [ ]:
gdf_florence = gpd.GeoDataFrame(
    florence,
    geometry=gpd.points_from_xy(florence["Long"], florence["Lat"]),
    crs="EPSG:4326",
)
print(gdf_florence.head(2))
print(gdf_florence.geom_type.unique())

**Q6)** The two layers together.

In [ ]:
fig, ax = plt.subplots(1, figsize=(30, 20))
base = lower48.plot(ax=ax, color="#3B3C6E")
gdf_florence.plot(ax=base, color="darkred", marker="*", markersize=10)
plt.show()

**Q7)** The coordinate reference systems.

In [ ]:
print("states:", country.crs)
print("track: ", gdf_florence.crs)

# Both are WGS84 geographic coordinates (EPSG:4326), so the two layers line up as plotted.
# Had they differed, one would have to be reprojected with .to_crs before overlaying.

**Q8)** Which states the track crosses.

In [ ]:
fig, ax = plt.subplots(figsize=(30, 20))
lower48.plot(ax=ax)

# annotate each state at its own centroid
for _, row in lower48.iterrows():
    ax.annotate(row["NAME"], xy=row.geometry.centroid.coords[0], ha="center")

# the overlay keeps only the track points that fall inside a state polygon
res_intersection = gdf_florence.overlay(country, how="intersection")
res_intersection.plot(ax=ax, color="red", marker="*", markersize=25)
plt.show()

print(res_intersection["NAME"].unique())